# 37 · Stage-3 training runs (additional experiments) — session A (T4 GPU; run sessions A and B at the same time)
190 runs, primary schedule only, about 13 min each: E1 faster-learning slot (6 models × 5 folds × seeds 0–2) and
E2 shared initialization of the trigonometric and identity controls (4 models × 5 folds × seeds 0–4). Cell 2 first runs a
pre-flight check of the new models (about 1 min) and stops if it fails.
Two copies of this notebook (A and B) share the work through lock files. Every run is trained in its own Python process
(`p2_run_stage3.py`, same code path as notebooks 13, 27 and 30), so GPU memory is released after each run (no out-of-memory build-up).
Each run is trained on the local disk of the runtime and copied to Drive once when it ends, so Drive Trash does not fill during training.
**Resumable:** after a disconnect simply **Run all** again; finished runs are skipped, an interrupted run restarts from epoch 0.
The prespecified text (cell 1) is stored with a UTC time stamp before the first run.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'A'
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, json, time, shutil, subprocess
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CODE     = f'{PROJECT}/code'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_stage3'
ROOT     = f'{RES}/p2_stage3_runs_train'
for d in (OUT, ROOT): os.makedirs(d, exist_ok=True)
PROTO = """Stage 3 - additional experiments (faster-learning bottleneck slot; shared initialization of the controls), written before any
stage-3 run was trained.

Motivation. In the five-seed analysis the 20 weights of the circuit and Dense(4) blocks moved little from their initial values under
the shared optimizer settings (Adam 1e-4), and the trigonometric and identity controls started from a different decoder
initialization than the circuit models of the same seed. Two additional experiments test whether these properties change the
conclusions. The five-seed results remain the main analysis; these experiments are reported as additional experiments, specified
before their runs.

A. Training runs. Code, data (400 CBIS-DDSM mass crops, folds.csv), settings and evaluation as for the five-seed runs: batch 16, Adam
   1e-4, up to 200 epochs, best checkpoint and early stopping (patience 10, from epoch 15) on the validation loss, learning rate halved
   after four epochs without improvement of the training loss (primary schedule 'train' only), Google Colab NVIDIA T4, one Python
   process per run (runner p2_run_stage3.py, models in qseg2_stage3.py). Run folders: results/p2_stage3_runs_train.
   E1 - faster-learning slot (90 runs): g_q_fast, g_cl_fast, g_tr_fast, p_q_fast, p_cl_fast, p_tr_fast; folds 0-4; seeds 0-2.
        Every weight of the 4->4 block is stored as v = theta / 100 and used as theta = 100 v, so the block learns with an effective
        learning rate 100 times that of the rest of the network (1e-2 at the start; it follows the schedule). All other layers and
        settings are unchanged. The initial effective block weights equal those of the standard model of the same seed; the
        trigonometric models start from the initial weights of the circuit model of the same placement and seed in every shared
        layer.
   E2 - shared initialization (100 runs): g_tr_shared, p_tr_shared, g_id_shared, p_id_shared; folds 0-4; seeds 0-4. Every layer
        shared with the circuit model of the same placement and seed (encoder, pre-slot, post-slot, decoder) starts from that
        model's initial weights; block weights and training settings are those of the standard g_tr, p_tr, g_id, p_id runs.
   Before training, every run whose initialization is copied checks that all shared layers are identical to the reference
   circuit model, and stops otherwise. Every run stores the initial and trained effective block weights and the change of the block
   and its neighbouring layers (pre-slot, post-slot, bottleneck_conv2, dec1_conv1). A run that stops because of an infrastructure
   error is restarted from epoch 0; a run whose validation loss is never finite is recorded as failed and reported. All 190 runs
   are reported; none is discarded or repeated for its result. No further stage-3 models, seeds, schedules or learning rates are
   added.

B. Test sets and evaluation: the five test sets and the evaluation code of the five-seed study (CBIS-DDSM test folds, frozen CBIS-DDSM
   cohort, DMID, CSAW-S with the consensus and the three per-radiologist references, CDD-CESM), threshold 0.5, per image Dice,
   IoU, HD95 and empty flag; per crop and seed, Dice is averaged over the five fold models on the sets other than the test folds.

C. Estimands and analysis (primary schedule; crossed hierarchical bootstrap over seeds, resampled jointly, and clusters - patients;
   DMID: source images - B = 10,000, numpy default_rng(0); verdicts as in the five-seed study: equivalent when the 95% CI lies within
   +-0.02 Dice, difference when it excludes zero, inconclusive otherwise).
   E1, per placement and test set, seeds 0-2: q_fast - cl_fast and q_fast - tr_fast (Holm within these four contrasts of each test
       set); q_fast - q, cl_fast - cl and tr_fast - tr (effect of the faster slot; descriptive); block weight change of each model.
   E2, per placement and test set, seeds 0-4: q - tr_shared and q - id_shared (Holm within these four contrasts of each test set),
       which replace the independent-seed approximation for these controls; tr_shared - tr and id_shared - id (effect of the
       initialization; descriptive). The standard q, cl, tr and id runs are those of the five-seed study.
   Reporting: if any q_fast - cl_fast or q_fast - tr_fast contrast on the CBIS-DDSM test folds favors the circuit with a 95% CI
   excluding zero, this is reported in the abstract; all other outcomes are reported in the results and the Online Resource."""
proto = f'{OUT}/PROTOCOL_stage3_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
assert open(proto).read().startswith(PROTO), 'protocol file on Drive differs from this notebook'
print(open(proto).read().splitlines()[-1])

In [ ]:
# 2 · GPU check (T4 only), runner script, local copy of the CBIS-DDSM crops
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
assert 'T4' in gpu, f'not a T4 ({gpu}): Runtime -> Change runtime type -> T4 GPU, then Run all'
for f in ('qseg2.py', 'qseg2_rev4.py'):
    assert os.path.exists(f'{CODE}/{f}'), f'{CODE}/{f} missing'
for f, src in (('qseg2_stage3.py', '"""Stage-3 additions to qseg2 (additional experiments, prespecified in protocols/PROTOCOL_stage3_additional_experiments.txt).\n\nE1 - faster-learning slot. The 4->4 block learns with an effective learning rate SLOT_SCALE (100) times that of the rest of the\n     network. This is done by reparameterization, so one Adam optimizer and the usual ReduceLROnPlateau schedule still apply:\n     every slot weight is stored as v = theta / k and used as theta = k * v. Adam moves v by about the learning rate per step,\n     so theta moves about k times as far. The initial theta is exactly that of the standard model of the same seed.\n     Models: g_q_fast, p_q_fast (circuit), g_cl_fast, p_cl_fast (Dense(4)), g_tr_fast, p_tr_fast (trigonometric).\nE2 - shared initialization. The trigonometric and identity controls start from the initial weights of the circuit model of the\n     same placement and seed in every layer they share with it (encoder, pre-slot, post-slot and decoder layers).\n     Models: g_tr_shared, p_tr_shared, g_id_shared, p_id_shared. The E1 trigonometric models are initialized the same way.\n\nLayer names are those of the standard models (quantum_circuit, classical_replacement, trig_replacement), so the standard\nevaluation code applies; the effective slot weights are SLOT_SCALE times the stored variables of the *_fast models.\nCall patch() once after importing qseg2 and qseg2_rev4 (qseg2_rev4.patch() is called here).\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\nimport qseg2_rev4\n\nSLOT_SCALE = 100.0\nSLOT_LAYERS = (\'quantum_circuit\', \'classical_replacement\', \'trig_replacement\')\nFAST = (\'g_q_fast\', \'p_q_fast\', \'g_cl_fast\', \'p_cl_fast\', \'g_tr_fast\', \'p_tr_fast\')\nSHARED = (\'g_tr_shared\', \'p_tr_shared\', \'g_id_shared\', \'p_id_shared\')\nNEEDS_SHARED_INIT = (\'g_tr_fast\', \'p_tr_fast\') + SHARED\nAUDIT_LAYERS = SLOT_LAYERS + (\'pre_dense\', \'pre_conv\', \'post_dense\', \'post_conv\', \'bottleneck_conv2\', \'dec1_conv1\')\n\n\ndef _shrink(layer, names):\n    for n in names:\n        v = getattr(layer, n)\n        v.assign(v / SLOT_SCALE)\n\n\nclass FastCircuit(qseg2.TFCircuitLayer):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'qweights\'])\n\n    def call(self, inputs):\n        from qseg.quantum_sim import circuit_expvals, tf_ops\n        return circuit_expvals(tf.cast(inputs, tf.float32), SLOT_SCALE * tf.cast(self.qweights, tf.float32), tf_ops(), self.n_qubits)\n\n\nclass FastPatchCircuit(FastCircuit):\n    def call(self, inputs):\n        x = tf.cast(inputs, tf.float32)\n        shape = tf.shape(x)\n        out = super().call(tf.reshape(x, [-1, self.n_qubits]))\n        return tf.reshape(out, shape)\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n\nclass FastDense(layers.Dense):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'kernel\', \'bias\'])\n\n    def call(self, inputs):\n        y = tf.tensordot(tf.cast(inputs, tf.float32), SLOT_SCALE * self.kernel, axes=[[-1], [0]]) + SLOT_SCALE * self.bias\n        return tf.tanh(y)\n\n\nclass FastTrig(qseg2_rev4.TrigLayer):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'A\', \'phi\'])\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * tf.cast(x, tf.float32) + SLOT_SCALE * self.phi)\n        return tf.tanh(tf.einsum(\'...i,ji->...j\', c, SLOT_SCALE * self.A))\n\n\n_prev_block = None\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    if kind == \'q_fast\':\n        return (FastPatchCircuit if patch else FastCircuit)(n_qubits, n_layers, name=\'quantum_circuit\')(z)\n    if kind == \'cl_fast\':\n        return FastDense(n_qubits, activation=\'tanh\', name=\'classical_replacement\')(z)\n    if kind == \'tr_fast\':\n        return FastTrig(n_qubits, name=\'trig_replacement\')(z)\n    if kind in (\'tr_shared\', \'id_shared\'):\n        return _prev_block(z, kind.split(\'_\')[0], n_qubits, n_layers, patch)\n    return _prev_block(z, kind, n_qubits, n_layers, patch)\n\n\n# ----------------------------------------------------------------------------- shared initialization\n_base_build = None\nINIT_SEED = None          # set by the runner before run_fold; consumed by the first model build\nINIT_RECORD = {}          # filled at that build: copy check and initial weights of the audited layers\n\n\ndef effective(layer, name):\n    """Effective weights of a layer as one flat float64 vector (slot of *_fast models: SLOT_SCALE x stored)."""\n    w = np.concatenate([v.numpy().astype(np.float64).ravel() for v in layer.trainable_weights]) if layer.trainable_weights else np.zeros(0)\n    if name in FAST and layer.name in SLOT_LAYERS:\n        w = w * SLOT_SCALE\n    return w\n\n\ndef build_p2_model(name, *args, **kwargs):\n    global INIT_SEED\n    model = _base_build(name, *args, **kwargs)\n    if INIT_SEED is None or name not in FAST + SHARED:\n        return model\n    seed, INIT_SEED = INIT_SEED, None\n    ref_name = name[:2] + \'q\'\n    qseg2.set_seeds(seed)\n    ref = _base_build(ref_name, *args, **kwargs)                  # standard circuit model of the same placement and seed\n    refw = {l.name: l.get_weights() for l in ref.layers if l.weights}\n    copied, before = [], {}\n    for l in model.layers:\n        if not l.weights or l.name not in refw:\n            continue\n        if l.name in SLOT_LAYERS:\n            if name.startswith((\'g_q\', \'p_q\')):                    # circuit: same initial angles as the standard model\n                w = refw[l.name]\n                l.set_weights([x / SLOT_SCALE for x in w] if name in FAST else w)\n            continue\n        a, b = l.get_weights(), refw[l.name]\n        if [x.shape for x in a] != [x.shape for x in b]:\n            continue\n        before[l.name] = float(max(np.abs(x - y).max() for x, y in zip(a, b)))\n        if name in NEEDS_SHARED_INIT:\n            l.set_weights(b); copied.append(l.name)\n    after = {l.name: float(max(np.abs(x - y).max() for x, y in zip(l.get_weights(), refw[l.name])))\n             for l in model.layers if l.weights and l.name in refw and l.name not in SLOT_LAYERS and l.name in before}\n    del ref\n    qseg2.set_seeds(seed)                                          # training randomness starts from the seed, as in all runs\n    INIT_RECORD.clear()\n    INIT_RECORD.update(reference=ref_name, seed=seed, copied_layers=copied, max_diff_before_copy=before,\n                       max_diff_after_copy=after, shared_identical=bool(all(v == 0 for v in after.values())),\n                       slot_scale=SLOT_SCALE if name in FAST else 1.0,\n                       init={l.name: effective(l, name) for l in model.layers if l.name in AUDIT_LAYERS})\n    return model\n\n\ndef weight_change(model, name):\n    """Change of the audited layers from INIT_RECORD[\'init\'] to the current (trained) weights."""\n    out = {}\n    for l in model.layers:\n        if l.name not in INIT_RECORD.get(\'init\', {}):\n            continue\n        a = INIT_RECORD[\'init\'][l.name]; b = effective(l, name); d = b - a\n        if l.name == \'quantum_circuit\':\n            d = (d + np.pi) % (2 * np.pi) - np.pi\n        out[l.name] = dict(n=int(a.size), max_abs_change=float(np.abs(d).max()), rms_change=float(np.sqrt((d ** 2).mean())),\n                           rel_change=float(np.linalg.norm(d) / max(np.linalg.norm(a), 1e-12)))\n    return out\n\n\ndef patch():\n    global _prev_block, _base_build\n    qseg2_rev4.patch()\n    if _prev_block is None:\n        _prev_block = qseg2._block\n        qseg2._block = _block\n    if _base_build is None:\n        _base_build = qseg2.build_p2_model\n        qseg2.build_p2_model = build_p2_model\n    lab = {\'q\': \'circuit\', \'cl\': \'Dense(4)\', \'tr\': \'trigonometric\', \'id\': \'identity\'}\n    for n in FAST:\n        qseg2.MODELS2.setdefault(n, f"{\'Global\' if n[0] == \'g\' else \'Patch-wise\'} bottleneck, {lab[n[2:].split(\'_\')[0]]}, slot learning rate x{SLOT_SCALE:g}")\n    for n in SHARED:\n        qseg2.MODELS2.setdefault(n, f"{\'Global\' if n[0] == \'g\' else \'Patch-wise\'} bottleneck, {lab[n[2:].split(\'_\')[0]]}, circuit-model initialization")\n    return qseg2\n'), ('check_stage3.py', '"""Pre-flight check of the stage-3 models (no data, a few seconds on a GPU). Stops with an error if any check fails.\n\n1. *_fast models: the effective initial block weights equal those of the standard model of the same seed, and the\n   output of a fast model equals that of the standard model on random input (the reparameterization changes nothing at start).\n2. Copied initialization (g_tr_fast, p_tr_fast, *_shared): every layer shared with the circuit model is identical.\n3. One Adam step (learning rate 1e-4) on the same batch moves the effective block weights of a fast model about 100 times\n   as far as those of the standard model.\n"""\nimport json, sys\nimport numpy as np\nimport tensorflow as tf\nimport qseg2\nimport qseg2_stage3 as S\n\nS.patch()\nSHAPE = (256, 256, 1)\nrng = np.random.default_rng(0)\nx = rng.random((2,) + SHAPE).astype(\'float32\'); y = (rng.random((2,) + SHAPE) > 0.7).astype(\'float32\')\nreport = {}\n\n\ndef build(name, seed, copy=True):\n    tf.keras.backend.clear_session()\n    qseg2.set_seeds(seed)\n    S.INIT_SEED = seed if copy else None\n    return qseg2.build_p2_model(name, SHAPE, 4, 2)\n\n\ndef slot(m, name):\n    for l in m.layers:\n        if l.name in S.SLOT_LAYERS:\n            return S.effective(l, name)\n\n\ndef one_step(m):\n    opt = tf.keras.optimizers.Adam(1e-4)\n    with tf.GradientTape() as t:\n        loss = qseg2.bce_dice_loss(y, m(x, training=False))\n    g = t.gradient(loss, m.trainable_weights)\n    opt.apply_gradients(zip(g, m.trainable_weights))\n\n\nfor p in (\'g\', \'p\'):\n    for blk in (\'q\', \'cl\'):\n        std, fast = f\'{p}_{blk}\', f\'{p}_{blk}_fast\'\n        ms = build(std, 1, copy=False); a0 = slot(ms, std); out_s = ms(x, training=False).numpy()\n        one_step(ms); da = np.abs(slot(ms, std) - a0).max()\n        mf = build(fast, 1); b0 = slot(mf, fast); out_f = mf(x, training=False).numpy()\n        one_step(mf); db = np.abs(slot(mf, fast) - b0).max()\n        rec = dict(init_max_diff=float(np.abs(a0 - b0).max()), output_max_diff=float(np.abs(out_s - out_f).max()),\n                   step_std=float(da), step_fast=float(db), ratio=float(db / max(da, 1e-12)),\n                   shared_identical=S.INIT_RECORD.get(\'shared_identical\'))\n        report[fast] = rec\n        assert rec[\'init_max_diff\'] < 1e-5, (fast, rec)\n        assert rec[\'output_max_diff\'] < 1e-4, (fast, rec)\n        assert 30 < rec[\'ratio\'] < 1000, (fast, rec)\n    for name in (f\'{p}_tr_fast\', f\'{p}_tr_shared\', f\'{p}_id_shared\'):\n        build(name, 1)\n        rec = dict(copied=len(S.INIT_RECORD[\'copied_layers\']), shared_identical=S.INIT_RECORD[\'shared_identical\'],\n                   differed_before=sum(v > 0 for v in S.INIT_RECORD[\'max_diff_before_copy\'].values()))\n        report[name] = rec\n        assert rec[\'shared_identical\'] and rec[\'copied\'] > 10, (name, rec)\n    tr_std = build(f\'{p}_tr\', 1, copy=False); t0 = slot(tr_std, f\'{p}_tr\')\n    trf = build(f\'{p}_tr_fast\', 1); assert np.abs(slot(trf, f\'{p}_tr_fast\') - t0).max() < 1e-5, \'trig fast init differs\'\n\nprint(json.dumps(report, indent=1))\nprint(\'STAGE-3 CHECK PASSED\')\n')):
    if os.path.exists(f'{CODE}/{f}'): assert open(f'{CODE}/{f}').read() == src, f'{CODE}/{f} differs from this notebook'
    else: open(f'{CODE}/{f}', 'w').write(src)
# qseg package: located exactly as in notebooks 13 and 27 (project code folder first, otherwise the first qseg/quantum_sim.py on Drive)
import glob, hashlib
QSEG_DIR = CODE
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
QSEG_MD5 = {os.path.basename(f): hashlib.md5(open(f, 'rb').read()).hexdigest() for f in sorted(glob.glob(f'{QSEG_DIR}/qseg/*.py'))}
json.dump(dict(qseg_dir=QSEG_DIR, md5=QSEG_MD5), open(f'{OUT}/qseg_used_{SESSION}.json', 'w'), indent=1)
print('qseg package:', QSEG_DIR, '| files:', len(QSEG_MD5))
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([QSEG_DIR, CODE] + ([os.environ['PYTHONPATH']] if os.environ.get('PYTHONPATH') else [])))
RUNNER = f'{CODE}/p2_run_stage3.py'
open(RUNNER, 'w').write('"""One training run of the stage-3 additional experiments (models of src/qseg2_stage3.py; same code path as src/p2_run.py).\nAfter training it writes init_record.json (shared-initialization check, slot scale) and weight_change.json\n(change of the slot and its neighbouring layers from initialization to the saved best weights).\n\nOriginal runner description:\n\n    python p2_run.py --model g_q --fold 0 --seed 3 --schedule train \\\n        --cbis /path/data/processed/cbis_mass_256_jit --out /path/results\n\nRuns go to <out>/p2_jit_runs_<schedule>/<model>_fold<k>_seed<s>/ (or to --root) exactly as in the notebooks (summary.json, history.csv,\nper-image CSV, best.weights.h5 without optimizer state). A finished run (summary.json present) is skipped, so a job array can\nsimply be resubmitted. Settings: Config defaults of the study with max_epochs=200; learning rate halved after 4 epochs without\nimprovement of the training loss (schedule \'train\') or of the validation loss (schedule \'val\'). The software and GPU of every\nrun are written to <run>/env.json.\n"""\nimport argparse, json, os, platform, subprocess, sys, time\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\n\nimport numpy as np\nimport tensorflow as tf\nimport qseg2\nimport qseg2_rev4\nimport qseg2_stage3\nfrom qseg.config import Config\n\nqseg2_stage3.patch()                       # adds g_tr / p_tr and the stage-3 models\n\n\ndef use_schedule(schedule):\n    """As notebook 27, cell 1: ReduceLROnPlateau watches \'loss\' (train) or \'val_loss\' (val)."""\n    base = tf.keras.callbacks.ReduceLROnPlateau\n    while getattr(base, \'_p2_switch\', False):\n        base = base.__mro__[1]\n    monitor = {\'train\': \'loss\', \'val\': \'val_loss\'}[schedule]\n\n    class SwitchPlateau(base):\n        _p2_switch = True\n\n        def __init__(self, monitor_=None, **kw):\n            kw.pop(\'monitor\', None)\n            super().__init__(monitor=monitor, **kw)\n\n    tf.keras.callbacks.ReduceLROnPlateau = SwitchPlateau\n\n\ndef shrink_weights(cfg, run_dir, name):\n    """As notebook 27, cell 2: weight file without optimizer state, verified weight by weight."""\n    big, small = f\'{run_dir}/best.weights.h5\', f\'{run_dir}/best_small.weights.h5\'\n    if not os.path.exists(big) or os.path.exists(f\'{run_dir}/.shrunk\'):\n        return\n    if os.path.getsize(big) < 60e6:\n        open(f\'{run_dir}/.shrunk\', \'w\').close(); return\n    shape = (cfg.image_size, cfg.image_size, cfg.in_channels)\n    tf.keras.backend.clear_session()\n    m1 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m1.load_weights(big); m1.save_weights(small)\n    m2 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m2.load_weights(small)\n    assert all(np.array_equal(a, b) for a, b in zip(m1.get_weights(), m2.get_weights())), f\'{run_dir}: weight check failed\'\n    os.replace(small, big); open(f\'{run_dir}/.shrunk\', \'w\').close()\n\n\ndef env_info():\n    try:\n        gpu = subprocess.run([\'nvidia-smi\', \'--query-gpu=name,driver_version\', \'--format=csv,noheader\'],\n                             capture_output=True, text=True, timeout=30).stdout.strip()\n    except Exception:\n        gpu = \'unknown\'\n    b = tf.sysconfig.get_build_info()\n    return dict(python=platform.python_version(), tensorflow=tf.__version__, keras=tf.keras.__version__ if hasattr(tf.keras, \'__version__\') else \'\',\n                cuda=b.get(\'cuda_version\'), cudnn=b.get(\'cudnn_version\'), gpu=gpu, host=platform.node())\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--model\', required=True, choices=sorted(qseg2.MODELS2))\n    ap.add_argument(\'--fold\', type=int, required=True, choices=range(5))\n    ap.add_argument(\'--seed\', type=int, required=True)\n    ap.add_argument(\'--schedule\', default=\'train\', choices=(\'train\', \'val\'))\n    ap.add_argument(\'--cbis\', required=True, help=\'folder with folds.csv, images/, masks/\')\n    ap.add_argument(\'--out\', required=True, help=\'results root\')\n    ap.add_argument(\'--root\', default=None, help=\'run folder root (default <out>/p2_jit_runs_<schedule>)\')\n    a = ap.parse_args()\n\n    root = a.root or os.path.join(a.out, f\'p2_jit_runs_{a.schedule}\')\n    run = f\'{a.model}_fold{a.fold}_seed{a.seed}\'\n    d = os.path.join(root, run)\n    if os.path.exists(os.path.join(d, \'summary.json\')):\n        print(\'already finished:\', run); return\n    os.makedirs(d, exist_ok=True)\n    json.dump(env_info(), open(os.path.join(d, \'env.json\'), \'w\'), indent=2)\n\n    use_schedule(a.schedule)\n    cfg = Config(work_dir=a.out, max_epochs=200)\n    if not os.path.exists(os.path.join(root, \'train_config.json\')):    # never overwrite the record of earlier runs\n        cfg.to_json(os.path.join(root, \'train_config.json\'))\n    if not os.path.exists(os.path.join(root, \'schedule.json\')):\n        json.dump({\'schedule\': a.schedule}, open(os.path.join(root, \'schedule.json\'), \'w\'))\n    data = qseg2.load_cbis(a.cbis)\n    t0 = time.time()\n    if a.model in qseg2_stage3.NEEDS_SHARED_INIT:                # check the shared initialization before training\n        qseg2_stage3.INIT_SEED = a.seed\n        qseg2.set_seeds(a.seed)\n        qseg2.build_p2_model(a.model, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)\n        if not qseg2_stage3.INIT_RECORD.get(\'shared_identical\') or not qseg2_stage3.INIT_RECORD.get(\'copied_layers\'):\n            raise RuntimeError(\'shared initialization check failed\')\n        tf.keras.backend.clear_session()\n    qseg2_stage3.INIT_SEED = a.seed\n    s = qseg2.run_fold(cfg, data, a.model, a.fold, a.seed, out_root=root, min_epochs=15, verbose=2)\n    rec = {k: v for k, v in qseg2_stage3.INIT_RECORD.items() if k != \'init\'}\n    rec[\'slot_init_effective\'] = {k: v.tolist() for k, v in qseg2_stage3.INIT_RECORD.get(\'init\', {}).items() if k in qseg2_stage3.SLOT_LAYERS}\n    json.dump(rec, open(os.path.join(d, \'init_record.json\'), \'w\'), indent=1)\n    shrink_weights(cfg, d, a.model)\n    tf.keras.backend.clear_session()\n    m = qseg2.build_p2_model(a.model, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)\n    m.load_weights(os.path.join(d, \'best.weights.h5\'))\n    json.dump(qseg2_stage3.weight_change(m, a.model), open(os.path.join(d, \'weight_change.json\'), \'w\'), indent=1)\n    print(f\'{run} [{a.schedule}]: Dice {s["dice_per_image"]:.4f}, {s["epochs_run"]} epochs, {(time.time() - t0) / 60:.1f} min\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
chk = f'{OUT}/preflight_check_{SESSION}.txt'
if not os.path.exists(chk) or 'PASSED' not in open(chk).read():
    res = subprocess.run([sys.executable, f'{CODE}/check_stage3.py'], capture_output=True, text=True, env=ENV)
    open(chk, 'w').write(res.stdout + res.stderr[-5000:])
    print(res.stdout[-3000:], res.stderr[-3000:])
    assert 'STAGE-3 CHECK PASSED' in res.stdout, 'pre-flight check failed: send the output above'
print('pre-flight check passed')
LOCAL = '/content/cbis_mass_256_jit'
if not os.path.exists(f'{LOCAL}/folds.csv'): shutil.copytree(CBIS_DIR, LOCAL)
print('crops:', len(os.listdir(f'{LOCAL}/images')), '| runner written')

In [ ]:
# 3 · Training (sessions share the work through lock files; one process per run)
E1 = ('g_q_fast', 'g_cl_fast', 'g_tr_fast', 'p_q_fast', 'p_cl_fast', 'p_tr_fast')
E2 = ('g_tr_shared', 'p_tr_shared', 'g_id_shared', 'p_id_shared')
RUNS = [(m, f, s, 'train') for s in (0, 1, 2) for f in range(5) for m in E1]
RUNS += [(m, f, s, 'train') for s in range(5) for f in range(5) for m in E2]
assert len(RUNS) == 190 and len(set(RUNS)) == 190
def root_of(m, sch): return ROOT
def rdir(m, f, s, sch): return f'{root_of(m, sch)}/{m}_fold{f}_seed{s}'
def finished(r): return os.path.exists(rdir(*r) + '/summary.json') or os.path.exists(rdir(*r) + '/failed.json')
LOCK_H = 2
def claim(r):
    d = rdir(*r)
    if finished(r): return False
    lk = f'{d}/.lock'
    if os.path.exists(lk):
        try: who, t = open(lk).read().split(); t = float(t)
        except Exception: who, t = SESSION, 0.0
        if who != SESSION and time.time() - t < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}'); time.sleep(30)
    try: return open(lk).read().split()[0] == SESSION
    except Exception: return False
WORK = '/content/p2_work'     # each run is trained on the local disk and copied to Drive once at the end:
                              # no checkpoint or history file is overwritten on Drive during training (keeps Drive Trash small)
def train(r):
    m, f, s, sch = r; log = f'/content/log_{m}_{f}_{s}_{sch}.txt'
    lroot = f'{WORK}/{os.path.basename(root_of(m, sch))}'; ldir = f'{lroot}/{m}_fold{f}_seed{s}'
    shutil.rmtree(ldir, ignore_errors=True); os.makedirs(lroot, exist_ok=True)
    cmd = [sys.executable, RUNNER, '--model', m, '--fold', str(f), '--seed', str(s), '--schedule', sch,
           '--cbis', LOCAL, '--out', RES, '--root', lroot]
    with open(log, 'w') as fh: rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=ENV).returncode
    d = rdir(*r); os.makedirs(d, exist_ok=True)
    if os.path.isdir(ldir):
        names = sorted(os.listdir(ldir), key=lambda n: n == 'summary.json')      # summary.json last = run complete on Drive
        for n in names:
            src = f'{ldir}/{n}'
            if os.path.isdir(src): shutil.copytree(src, f'{d}/{n}', dirs_exist_ok=True)
            else: shutil.copy2(src, f'{d}/{n}')
        shutil.rmtree(ldir, ignore_errors=True)
    shutil.copy(log, d + f'/train_log_attempt{int(time.time())}.txt')
    return rc, open(log).read()[-3000:]
order = RUNS if SESSION == 'A' else RUNS[::-1]
for r in order:
    if not claim(r): continue
    m, f, s, sch = r; t0 = time.time()
    rc, tail = train(r)
    if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
        if 'validation loss never finite' in tail:
            json.dump(dict(model=m, fold=f, seed=s, schedule=sch, failed='validation loss never finite'), open(rdir(*r) + '/failed.json', 'w'))
        else:
            print(f'   {m} f{f} s{s} [{sch}]: process error (code {rc}), retrying once from epoch 0 ...\n' + tail[-1500:], flush=True)
            rc, tail = train(r)
            if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
                print('   second failure - stopping this session so that the error can be checked:\n' + tail, flush=True)
                raise SystemExit(1)
    done = sum(os.path.exists(rdir(*q) + '/summary.json') for q in RUNS)
    if os.path.exists(rdir(*r) + '/summary.json'):
        S = json.load(open(rdir(*r) + '/summary.json'))
        print(time.strftime('%H:%M UTC', time.gmtime()), f"| {m} fold {f} seed {s} [{sch}]: Dice {S['dice_per_image']:.4f}, "
              f"{S['epochs_run']} ep (best {S['best_epoch']}), {(time.time() - t0) / 60:.1f} min | finished {done} / {len(RUNS)}", flush=True)
    else:
        print(time.strftime('%H:%M UTC', time.gmtime()), f'| {m} fold {f} seed {s} [{sch}]: FAILED (recorded) | finished {done} / {len(RUNS)}', flush=True)
left = [r for r in RUNS if not finished(r)]
print(f'session {SESSION} has nothing left to start; {len(left)} run(s) still running in the other session')

In [ ]:
# 4 · Overview of the stage-3 runs (with block weight change)
import pandas as pd
rows = [dict(json.load(open(rdir(*r) + '/summary.json')), schedule=r[3]) for r in RUNS if os.path.exists(rdir(*r) + '/summary.json')]
for r, row in zip([r for r in RUNS if os.path.exists(rdir(*r) + '/summary.json')], rows):
    wc = rdir(*r) + '/weight_change.json'
    if os.path.exists(wc):
        W = json.load(open(wc))
        for ln in ('quantum_circuit', 'classical_replacement', 'trig_replacement'):
            if ln in W: row.update(slot_max_change=W[ln]['max_abs_change'], slot_rel_change=W[ln]['rel_change'])
ov = pd.DataFrame(rows); ov.to_csv(f'{OUT}/overview_stage3_{SESSION}.csv', index=False)
print(len(ov), 'of', len(RUNS), 'finished;', sum(os.path.exists(rdir(*r) + '/failed.json') for r in RUNS), 'failed')
if len(ov): print(ov.groupby(['model'])[[c for c in ('dice_per_image', 'epochs_run', 'best_epoch', 'slot_max_change', 'slot_rel_change') if c in ov]].mean().round(4).to_string())